## Setup PyTorch and NLTK with Data

In [1]:
%pip install nltk
import torch
import time
import nltk
import numpy as np
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader

In [2]:
document = """The environment is an intricate web of life that provides the foundation for all living beings. It encompasses everything from forests and oceans to the air we breathe and the soil beneath our feet. Human activities, however, have disrupted this delicate balance through pollution, deforestation, excessive resource consumption, and greenhouse gas emissions. These actions have led to global warming, extreme weather events, and the loss of biodiversity. Protecting the environment is not just about saving trees or endangered species—it’s about safeguarding human health, preserving ecosystems, and ensuring future generations inherit a livable planet. By adopting sustainable practices, supporting conservation efforts, and raising awareness about environmental issues, we can all play a role in restoring this balance. The environment is not separate from us; it is us. Our survival depends on how well we care for the natural world that surrounds and supports us."""


## Tokenization and Vocabulary

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
tokens = word_tokenize(document.lower())
vocab = {'<unk>':0}
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)
print(len(vocab))
vocab

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


111


{'<unk>': 0,
 'the': 1,
 'environment': 2,
 'is': 3,
 'an': 4,
 'intricate': 5,
 'web': 6,
 'of': 7,
 'life': 8,
 'that': 9,
 'provides': 10,
 'foundation': 11,
 'for': 12,
 'all': 13,
 'living': 14,
 'beings': 15,
 '.': 16,
 'it': 17,
 'encompasses': 18,
 'everything': 19,
 'from': 20,
 'forests': 21,
 'and': 22,
 'oceans': 23,
 'to': 24,
 'air': 25,
 'we': 26,
 'breathe': 27,
 'soil': 28,
 'beneath': 29,
 'our': 30,
 'feet': 31,
 'human': 32,
 'activities': 33,
 ',': 34,
 'however': 35,
 'have': 36,
 'disrupted': 37,
 'this': 38,
 'delicate': 39,
 'balance': 40,
 'through': 41,
 'pollution': 42,
 'deforestation': 43,
 'excessive': 44,
 'resource': 45,
 'consumption': 46,
 'greenhouse': 47,
 'gas': 48,
 'emissions': 49,
 'these': 50,
 'actions': 51,
 'led': 52,
 'global': 53,
 'warming': 54,
 'extreme': 55,
 'weather': 56,
 'events': 57,
 'loss': 58,
 'biodiversity': 59,
 'protecting': 60,
 'not': 61,
 'just': 62,
 'about': 63,
 'saving': 64,
 'trees': 65,
 'or': 66,
 'endangered': 67

In [4]:
input_sentences = document.split('\n')

def text_to_indices(sentence, vocab):
  numerical_sentence = []
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])
  return numerical_sentence

In [5]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))
len(input_numerical_sentences)


1

In [6]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])
len(training_sequence)

165

In [7]:
training_sequence[:5]
len_list = []
for sequence in training_sequence:
  len_list.append(len(sequence))
print(max(len_list))
training_sequence[0]

166


[1, 2]

In [8]:
padded_training_sequence = []
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)
len(padded_training_sequence[10])
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   1,  ..., 109,  22, 110],
        [  0,   1,   2,  ...,  22, 110,  99],
        [  1,   2,   3,  ..., 110,  99,  16]])

In [9]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]
print(X,y)

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   1,  ...,   9, 109,  22],
        [  0,   1,   2,  ..., 109,  22, 110],
        [  1,   2,   3,  ...,  22, 110,  99]]) tensor([  2,   3,   4,   5,   6,   7,   8,   9,  10,   1,  11,  12,  13,  14,
         15,  16,  17,  18,  19,  20,  21,  22,  23,  24,   1,  25,  26,  27,
         22,   1,  28,  29,  30,  31,  16,  32,  33,  34,  35,  34,  36,  37,
         38,  39,  40,  41,  42,  34,  43,  34,  44,  45,  46,  34,  22,  47,
         48,  49,  16,  50,  51,  36,  52,  24,  53,  54,  34,  55,  56,  57,
         34,  22,   1,  58,   7,  59,  16,  60,   1,   2,   3,  61,  62,  63,
         64,  65,  66,  67,  68,  69,  70,  63,  71,  32,  72,  34,  73,  74,
         34,  22,  75,  76,  77,  78,  79,  80,  81,  16,  82,  83,  84,  85,
         34,  86,  87,  88,  34,  22,  89,  90,  63,  91,  92,  34,  26,  93,
        

## Dataset Class

In [10]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [11]:
dataset = CustomDataset(X,y)
len(dataset)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## Long-Short Term Memory Structure

In [12]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [13]:
model = LSTMModel(len(vocab))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LSTMModel(
  (embedding): Embedding(111, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=111, bias=True)
)

## Training, Prediction and Evaluation

In [14]:
epochs = 50
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for epoch in range(epochs):
  total_loss = 0
  for batch_x, batch_y in dataloader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    optimizer.zero_grad()
    output = model(batch_x)
    loss = criterion(output, batch_y)
    loss.backward()
    optimizer.step()
    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 28.2047
Epoch: 2, Loss: 27.3914
Epoch: 3, Loss: 26.8659
Epoch: 4, Loss: 26.1342
Epoch: 5, Loss: 25.0034
Epoch: 6, Loss: 23.5158
Epoch: 7, Loss: 22.0473
Epoch: 8, Loss: 20.6930
Epoch: 9, Loss: 19.0618
Epoch: 10, Loss: 17.3149
Epoch: 11, Loss: 15.7376
Epoch: 12, Loss: 14.4897
Epoch: 13, Loss: 13.2032
Epoch: 14, Loss: 11.8444
Epoch: 15, Loss: 10.3547
Epoch: 16, Loss: 9.2684
Epoch: 17, Loss: 8.1507
Epoch: 18, Loss: 7.4801
Epoch: 19, Loss: 6.4227
Epoch: 20, Loss: 5.8186
Epoch: 21, Loss: 5.2217
Epoch: 22, Loss: 4.8544
Epoch: 23, Loss: 4.3243
Epoch: 24, Loss: 3.9458
Epoch: 25, Loss: 3.3910
Epoch: 26, Loss: 3.0963
Epoch: 27, Loss: 3.0433
Epoch: 28, Loss: 2.5602
Epoch: 29, Loss: 2.5578
Epoch: 30, Loss: 2.1861
Epoch: 31, Loss: 2.0614
Epoch: 32, Loss: 1.9500
Epoch: 33, Loss: 1.8111
Epoch: 34, Loss: 1.6351
Epoch: 35, Loss: 1.4855
Epoch: 36, Loss: 1.4359
Epoch: 37, Loss: 1.2946
Epoch: 38, Loss: 1.3154
Epoch: 39, Loss: 1.1978
Epoch: 40, Loss: 1.1193
Epoch: 41, Loss: 1.0379
Epoch: 42,

In [15]:
# // prediction

def prediction(model, vocab, text):
  # tokenize
  tokenized_text = word_tokenize(text.lower())
  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)
  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)
  # send to model
  output = model(padded_text)
  # predicted index
  value, index = torch.max(output, dim=1)
  # merge with text
  return text + " " + list(vocab.keys())[index]
prediction(model, vocab, "The course follows a monthly")


'The course follows a monthly environment'

In [16]:
num_tokens = 10
input_text = "hi how are"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)

hi how are environment
hi how are environment is
hi how are environment is an
hi how are environment is an intricate
hi how are environment is an intricate web
hi how are environment is an intricate web of
hi how are environment is an intricate web of life
hi how are environment is an intricate web of life that
hi how are environment is an intricate web of life that provides
hi how are environment is an intricate web of life that provides the


In [ ]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

# // Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 100.00%
